#            
##             Semantic Segmentation with Deep Learning

---

**         **
             :
-         (     )
-   **     **          
-     **         **

**           :**
- Dataset        
-                  
-   CNN            
-   U-Net         CNN      

---
> **     :**   Markdown                     (  ** **).
>            !"


---
##      :     (Semantic Segmentation)  

###          ...

                 .    :
**                 **        

      **   **   Semantic Segmentation  .

###         (Object Detection):
- **Object Detection**:             "       "
- **Semantic Segmentation**:     **   **     "               "

```
 :                    :    
                 
   (  RGB)           AI         =       
                                       =        
                                       =     
                 
```

###      
-       (     / / )
-     (     /   )
-     (       )


---
##      : Dataset      

###   Dataset  

** **        .   ISPRS            
(      =        )                
** **      .             Dataset      .

###    :   GeoTIFF  

GeoTIFF       **   **  .                  
(RGB)  **   **  :

```
 
      : Red ( )                        
      : Green ( )                        
      : Blue ( )                         
      : Infrared ( )           
      : Elevation ( )            
      : Labels ( )                !  
 
```

###      :
|   |     |     |
|---|---|---|
| 0 | Impervious surface (   :    ) |     |
| 1 | Building ( ) |     |
| 2 | Tree ( ) |     |
| 3 | Low vegetation (   ) |    -  |
| 4 | Car ( ) |     |
| 5 | Clutter/Background ( ) |     |

> ** :**                  .
>                          !


---
##            

###          

- **`rasterio`**:     GeoTIFF (  PIL        )
- **`numpy`**:           (    =      )
- **`matplotlib`**:          
- **`scikit-learn`**:   ML     KFold split
- **`tensorflow`**:                  

>              .   `Successfully installed`        .


In [ ]:
import os, random, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import KFold
from IPython.display import Image, display

matplotlib.use('Agg')  # headless mode
print('All libraries loaded OK')
print('TensorFlow version:', tf.__version__)
print('Rasterio version:  ', rasterio.__version__)

---
##         Import        

###          

          notebook            .

**     :**
```python
import os           #           (  os.path.join)
import random       #     (  random.choice)
import json         #         JSON
import numpy as np  #           np   numpy  
import matplotlib.pyplot as plt  #        
import rasterio     #   GeoTIFF
import tensorflow as tf          #    
from tensorflow import keras     # API   TensorFlow
from tensorflow.keras import layers  #      
```

### SEED  
`SEED = 42`            .
  SEED                .        !

### DATA_DIR  
        `.tif` (  dataset)      .
    dataset                    .


In [ ]:
import os, random, json
import numpy as np
print('Basic imports loaded OK')

In [ ]:
import matplotlib
matplotlib.use('Agg')  # headless mode
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
print('Matplotlib loaded OK')

In [ ]:
import rasterio
print('Rasterio loaded OK')

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint
print(f'TensorFlow {tf.__version__} loaded OK')

In [ ]:
from sklearn.model_selection import KFold
from IPython.display import Image, display
print('All imports ready OK')

In [ ]:
# \u2500\u2500 Reproducibility seed \u2500\u2500
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# \u2500\u2500 Global settings \u2500\u2500
N_FOLDS      = 5
NUM_CLASSES  = 6
NUM_SAMPLES  = 100   # For full training on Kaggle, use 5000
BATCH_SIZE   = 8
EPOCHS       = 20
LEARNING_RATE = 1e-4
OUTPUT_DIR   = './outputs'
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, 'best_model.h5')

if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

CLASS_NAMES = [
    'Impervious surface', 'Building', 'Tree',
    'Low vegetation', 'Car', 'Clutter/Background'
]
CLASS_COLORS = [
    [255, 255, 255],  # white
    [0,   0,   255],  # blue
    [0,   255,   0],  # green
    [0,   255, 255],  # cyan
    [255, 255,   0],  # yellow
    [255,   0,   0],  # red
]

def discover_data_dir():
    """Auto-locate the folder containing .tif files."""
    candidates = [
        '/kaggle/input/potsdam-geotif/Potsdam-GeoTif',  # Kaggle path
        './PROJECT/Potsdam-GeoTif/Potsdam-GeoTif',
        '../PROJECT/Potsdam-GeoTif/Potsdam-GeoTif',
        './Potsdam-GeoTif/Potsdam-GeoTif',
        './data'
    ]
    for p in candidates:
        if os.path.isdir(p):
            if any(f.endswith('.tif') for f in os.listdir(p)): return p
    
    # Fallback search
    for root, _, files in os.walk('.'):
        if any(f.endswith('.tif') for f in files): return root
    return 'data'

DATA_DIR = discover_data_dir()
print(f"Data directory: {DATA_DIR}")
print(f"Seed initialized: {SEED}")

---
#  
#    :   Dataset
#  

##      :
 .             dataset
 .                
 .           (fold)      

##          

          AI     **     **.
                         !


###  .          GeoTIFF

####   `get_all_tif_files`      

```python
os.walk(data_dir)
```
        **   **                
             .

```python
if f.endswith('.tif')
```
        `.tif`        .

```python
tif_files.append(os.path.join(root, f))
```
      (  +    )          .

>   dataset  :          ,       .
>         sample            .


In [ ]:
def get_all_tif_files(data_dir):
    """Recursively find all .tif file paths."""
    tif_files = []
    for root, _, files in os.walk(data_dir):
        for f in files:
            if f.endswith('.tif'):
                tif_files.append(os.path.join(root, f))
    return sorted(tif_files)

all_files_raw = get_all_tif_files(DATA_DIR)
all_files = all_files_raw[:NUM_SAMPLES]

print(f'Total GeoTIFF files found: {len(all_files_raw)}')
print(f'Files selected for this session: {len(all_files)}')
print('Sample files:', [os.path.basename(f) for f in all_files[:5]])

###  .             

####   `rasterio.open`  

`rasterio`   `PIL.Image.open`                .

```python
with rasterio.open(file_path) as src:
    data = src.read()  #           numpy     (6, H, W)
```

   : `(6, 224, 224)`  :
- ** **:    
- **224**:     ( )  
- **224**:     ( )

####   `normalize_band`          

       :          .
`matplotlib`          -     -   .
                     :

```
  = (  -  ) / (  -  )
 : (300 - 100) / (500 - 100) = 200/400 = 0.5
```

####   `label_to_rgb`  

             -    (       ).
                   :
-                   ...


In [ ]:
#        

def normalize_band(band):
    """
                [0, 1]    .
    
                matplotlib
    (matplotlib   float  :   0.0   1.0  )
    
     : normalized = (x - min) / (max - min)
    """
    b_min = band.min()  #    
    b_max = band.max()  #    
    if b_max == b_min:  #           (         )
        return np.zeros_like(band, dtype=np.float32)
    return (band - b_min).astype(np.float32) / (b_max - b_min)


def label_to_rgb(label_band, colors):
    """
        (  0-5)            .
    
     : label_band     (H, W)     0-5
           colors           [R, G, B]
     : rgb     (H, W, 3)    
    """
    h, w = label_band.shape          #        
    rgb = np.zeros((h, w, 3), dtype=np.uint8)  #       ( )
    
    for class_idx, color in enumerate(colors):
        #                    
        mask = (label_band == class_idx)
        #            
        rgb[mask] = color
    
    return rgb


#            

#      
EXCLUDED = '0000000224-0000042784.tif'  #          
candidates = [f for f in all_files if EXCLUDED not in f]
sample_file = random.choice(candidates) if candidates else all_files[0]
print('     :', os.path.basename(sample_file))

#       rasterio
with rasterio.open(sample_file) as src:
    data = src.read()           #  : (6, H, W)
    crs  = src.crs              #      
    transform = src.transform   #    

print(f'\n   : {data.shape}')
print(f'    {data.shape[0]}   | {data.shape[1]}     | {data.shape[2]}    ')
print(f'   : {crs}')
print(f'\n       :')
band_names = ['Red','Green','Blue','Infrared','Elevation','Labels']
for i, name in enumerate(band_names):
    print(f'    {i} ({name}): {data[i].min()}   {data[i].max()}')


###  .           

####        
- **  RGB**:            
- **   **:            
- **   **:                  

#### `np.stack`  
     D                D (   )  :
```
stack([R(224,224), G(224,224), B(224,224)], axis=-1)   (224,224,3)
```

#### `plt.subplots(1, 3)`  
  figure   **   **   **   **            .

#### `plt.colorbar`  ?
                         .
     :     =    


In [ ]:
#            

#      
red   = data[0].astype(np.float32)   #    
green = data[1].astype(np.float32)   #    
blue  = data[2].astype(np.float32)   #    
elev  = data[4].astype(np.float32)   #    
label = data[5].astype(np.int32)     #     (   )

#     RGB:            
# axis=-1                  
rgb_img = np.stack([
    normalize_band(red),    # R     0.0   1.0
    normalize_band(green),  # G     0.0   1.0
    normalize_band(blue),   # B     0.0   1.0
], axis=-1)  #  : (H, W, 3)

#      
elev_normalized = normalize_band(elev)  #  : (H, W)

#        
label_rgb = label_to_rgb(label, CLASS_COLORS)  #  : (H, W, 3)

#   legend (   )
patches = [
    mpatches.Patch(
        color=[c/255 for c in CLASS_COLORS[i]],  #       0-1
        label=CLASS_NAMES[i]                      #    
    )
    for i in range(NUM_CLASSES)
]

#              
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('        Dataset  ', fontsize=14, fontweight='bold')

#    : RGB
axes[0].imshow(rgb_img)                   #     (H,W,3)      
axes[0].set_title('  RGB (  0,1,2)', fontsize=11)
axes[0].axis('off')                       #      

#    :     colorbar
im = axes[1].imshow(elev_normalized, cmap='terrain')  # cmap        
axes[1].set_title('    (Elevation)', fontsize=11)
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label='   ')

#    :  
axes[2].imshow(label_rgb)
axes[2].set_title('    (Label Map)', fontsize=11)
axes[2].axis('off')
axes[2].legend(handles=patches, loc='lower right', fontsize=7, framealpha=0.9)

plt.tight_layout()
out_vis = os.path.join(DATA_DIR, 'step1_visualization.png')
plt.savefig(out_vis, dpi=150, bbox_inches='tight')
plt.close()
display(Image(out_vis))
print('     :', out_vis)


###  .         

####          

                 .
                    ** **  !

              dataset ** **      .
    dataset            .

#### `(label == i).sum()`      
- `label == i`       True/False       label
- `.sum()`     True  =       i


In [ ]:
#        

print('           :')
print(f'{" ":<30} {" ":>12} {" ":>8} {" "}')
print('=' * 70)

total_pixels = label.size  #       (H   W)

for i, name in enumerate(CLASS_NAMES):
    count   = (label == i).sum()              #        
    percent = 100 * count / total_pixels      #      
    bar     = '|' + ' ' * int(percent / 2)   #    
    print(f'{name:<30} {count:>12,} {percent:>7.2f}%  {bar}')

print('=' * 70)
print(f'   : {total_pixels:,}  (= {data.shape[1]}   {data.shape[2]})')

#    
dominant = max(range(NUM_CLASSES), key=lambda i: (label==i).sum())
print(f'\n   :  {CLASS_NAMES[dominant]}    {100*(label==dominant).sum()/total_pixels:.1f}%')


###  .      K-Fold

####  :          

                **   **   ** **    :
                         .
        **     !**

####  : K-Fold Cross-Validation

        **K    ** (fold)    :

```
Dataset:   (100  )
            Fold 1   Fold 2   Fold 3   Fold 4   Fold 5  
           Training (60%)        Validation (20%)         Test (20%)
              Fold 1+2+3             Fold 4                Fold 5
```

- **Train (Fold 1+2+3)**:          
- **Validation (Fold 4)**:               (     )
- **Test (Fold 5)**:                  

#### `KFold(n_splits=5)`  
`scikit-learn`            :
-   (0   N-1)              
- `shuffle=True`               (   )


In [ ]:
#   Step 1c: K-Fold Cross-Validation Split  

# With only a few files, repeat them to demonstrate the split mechanism
demo_files = all_files * max(1, (N_FOLDS * 4) // max(len(all_files), 1) + 1)
demo_files = demo_files[:max(len(all_files), N_FOLDS * 4)]

kf  = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
arr = np.array(demo_files)
folds = [arr[idx].tolist() for _, idx in kf.split(arr)]

train_files = folds[0] + folds[1] + folds[2]   # Folds 1, 2, 3   Training
val_files   = folds[3]                            # Fold 4          Validation
test_files  = folds[4]                            # Fold 5          Test

print('K-Fold split result:')
for i, fold in enumerate(folds, 1):
    role = 'Train' if i<=3 else ('Val' if i==4 else 'Test')
    print(f'  Fold {i} ({role:5}): {len(fold)} files')
print(f'\nTotal  Train  : {len(train_files)} files')
print(f'Total  Val    : {len(val_files)} files')
print(f'Total  Test   : {len(test_files)} files')

splits_path = os.path.join(DATA_DIR, 'fold_splits.json')
with open(splits_path, 'w') as fp:
    json.dump({'train':train_files,'val':val_files,'test':test_files,'all_folds':folds}, fp, indent=2)
print('\nSaved fold_splits.json')

###      

**     **
-   GeoTIFF          
-        
-            
- K-Fold Cross-Validation      

**       **
- `step1_visualization.png`      
- `fold_splits.json`     dataset

**         :          !**


---
#  
#    :           (CNN)
#  

##    :       (CNN)  

###     CNN:      

          (Fully Connected):
-   224 224      ,   
-              : **     **   **   **!
-         overfit  

###  :   (Convolution)

** **         (  3 3)         ** **  :

```
   :             3 3:            (Feature Map):
                  
  1  2  3  4  5         1 0 -1                    ...   
  6  7  8  9  10        2 0 -2             ...              
  11 12 13 14 15        1 0 -1                               
  ...                               
 
```

** :**
-   3 3   **9  **   (   )
-               (Local Pattern Learning)
-          

###    : Fully Convolutional Network

```
  (H W 4)
     
[Conv 3 3   32] + BN + ReLU       1:    
[Conv 3 3   32] + BN + ReLU
     
[Conv 3 3   64] + BN + ReLU       2:    
[Conv 3 3   64] + BN + ReLU
     
[Conv 3 3   128] + BN + ReLU      3:    
[Conv 3 3   128] + BN + ReLU
     
[Conv 1 1   6]                      :     (   )
     
Softmax                                
  (H W 6)
```

**   :**   MaxPooling            .
                   .


###      

#### BatchNormalization  
                   .
BatchNorm     ** **     ( ~   ~ ).

 :          .

#### ReLU (Rectified Linear Unit)  

```
ReLU(x) = max(0, x)
```
-          
-          

        ** **  .   ReLU         
            (       ).

#### Softmax  

```
softmax([2.0, 1.0, 0.5]) = [0.66, 0.24, 0.10]
```
      ** **     (  =  ).
     :                .

#### Conv2D(1 1)  
          ** **  .
128       6                .


###  .         

####              

  dataset     (   )            RAM  .
      (batch)        .

####   `load_sample`      
 .   GeoTIFF      
 .           (RGB+IR   RGB+IR+Elevation)
 .           (       )
 .       **One-Hot**    

#### One-Hot Encoding  

        (  4    )    ** **      :
```
  4 ( )   [0, 0, 0, 0, 1, 0]
  2 ( )    [0, 0, 1, 0, 0, 0]
```
      (Categorical Cross-Entropy)          .

#### `data.transpose(1,2,0)`  
rasterio         `(bands, H, W)`  .
TensorFlow         `(H, W, bands)`  .
`transpose(1,2,0)`          :
```
(6, 224, 224)   (224, 224, 6)
```


In [ ]:
#          
BATCH_SIZE = 2       #           (batch)
                     #   GPU     (  16   32)
EPOCHS_CNN = 20      #      
LR_CNN     = 1e-3    #     (Learning Rate): 0.001
                     #              

def load_sample(file_path, use_elevation=False):
    """
        GeoTIFF            .
    
     :
        file_path     :     .tif
        use_elevation :   True   5   (RGB+IR+Elev)
                          False   4   (RGB+IR)
     :
        X :     (H, W, channels)
        y :   One-Hot (H, W, 6)
    """
    with rasterio.open(file_path) as src:
        d = src.read()  #  : (6, H, W)
    
    #      
    n_bands = 5 if use_elevation else 4
    
    #          : (6,H,W)   (H,W,n_bands)
    X = d[:n_bands].transpose(1, 2, 0).astype(np.float32)
    
    #           [0,1]
    for c in range(X.shape[-1]):
        X[..., c] = normalize_band(X[..., c])
    
    #   One-Hot: (H,W)   (H,W,6)
    label_band = d[5].astype(np.int32)
    y = tf.keras.utils.to_categorical(label_band, num_classes=NUM_CLASSES)
    
    return X, y

#          
X4, y = load_sample(sample_file, use_elevation=False)  # 4  
X5, _ = load_sample(sample_file, use_elevation=True)   # 5  
print('    4   (RGB+IR)          :', X4.shape)
print('    5   (RGB+IR+Elevation):', X5.shape)
print('    One-Hot                   :', y.shape)
print('\n  One-Hot     (0,0):', y[0,0,:])
print('     :', y[0,0,:].argmax(), '=', CLASS_NAMES[y[0,0,:].argmax()])


###  .      Augmentation (   )

####   Overfitting  

**Overfitting** =         ** **     ** **  .

 :                      .

**  Overfitting:**
- Training Accuracy     (  99%)
- Validation Accuracy   (  55%)
-  : Training Loss       Validation Loss    

#### Data Augmentation:  

            **   **  .
                 .

```
   :    flip  :     flip  :      90 :
                
                                      
                                  ...      
                
```

** :**              ! (X   y     flip  )


In [ ]:
def augment(X, y):
    """
                     .
    
     :       X   y        
         - -     .
    
     : X   (H, W, C)  |  y   (H, W, 6)
     :            
    """
    # flip   (         )
    # [:, ::-1, :]          
    if np.random.rand() > 0.5:
        X = X[:, ::-1, :]   #         X
        y = y[:, ::-1, :]   #       y
    
    # flip   (         )
    # [::-1, :, :]          
    if np.random.rand() > 0.5:
        X = X[::-1, :, :]   #         X
        y = y[::-1, :, :]   #       y
    
    #    : 0  90  180    270 
    k = np.random.randint(0, 4)  #     0   3
    # np.rot90:     k 90    
    X = np.rot90(X, k).copy()   # .copy()      
    y = np.rot90(y, k).copy()
    
    return X, y


def make_dataset(file_list, augment_data=False, batch_size=2, use_elevation=False):
    """
    Build a tf.data.Dataset from a list of tile file paths using a generator.
    
    This version streams data from disk instead of loading everything into RAM,
    preventing OutOfMemory (OOM) errors and speeding up training significantly.
    """
    def generator():
        indices = np.arange(len(file_list))
        if augment_data:
            np.random.shuffle(indices)
            
        for idx in indices:
            X, y = load_sample(file_list[idx], use_elevation=use_elevation)
            if augment_data:
                X, y = augment(X, y)
            yield X, y

    n_bands = 5 if use_elevation else 4
    output_signature = (
        tf.TensorSpec(shape=(128, 128, n_bands), dtype=tf.float32),
        tf.TensorSpec(shape=(128, 128, 6), dtype=tf.float32)
    )

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=output_signature
    )

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

print('Building optimized datasets (streaming mode) ...')
train_ds = make_dataset(train_files, augment_data=True,  batch_size=BATCH_SIZE)
val_ds   = make_dataset(val_files,   augment_data=False, batch_size=BATCH_SIZE)
test_ds  = make_dataset(test_files,  augment_data=False, batch_size=BATCH_SIZE)
print('  Training dataset   : ready (streaming)')
print('  Validation dataset : ready (streaming)')
print('  Test dataset       : ready (streaming)')


###  .          CNN

#### `keras.Input`  
     . `shape=(None, None, 4)`  :
- `None, None`:         ( )
- `4`:       (RGB + IR)

#### `layers.Conv2D(32, 3, padding='same')`  
- `32`:     (           )
- `3`:     (3 3)
- `padding='same'`:                   =  

####         (32   64   128)?

          (   ).
        (       ).
               .

#### `activation='relu'`  
ReLU         Conv2D           ReLU    .


In [ ]:
def build_simple_cnn(input_channels=4, num_classes=6):
    """
      Simple CNN        .
    
     : Fully Convolutional Network
     : (H, W, input_channels)
     : (H, W, num_classes)              
    """
    #      
    inp = keras.Input(shape=(None, None, input_channels), name='input')
    
    #      :     (32  )  
    #                      
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)  #  
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    #      :     (64  )  
    #                    
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    #      :     (128  )  
    #                    
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    #        
    # Conv 1 1: 128       6   (   )    
    # softmax:         (  = 1      )
    out = layers.Conv2D(num_classes, 1, padding='same',
                        activation='softmax', name='output')(x)
    
    model = keras.Model(inputs=inp, outputs=out, name='SimpleCNN')
    return model


#    
cnn = build_simple_cnn(input_channels=4, num_classes=NUM_CLASSES)

#   optimizer      
# Adam:   optimizer        
# categorical_crossentropy:            
cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_CNN),
    loss='categorical_crossentropy',
    metrics=['accuracy']  #      
)

#    
print('    Simple CNN:')
cnn.summary()
print(f'\n   : {cnn.count_params():,}')


###  .       

####          

    **epoch** ( ):
1.   **batch** (   )   training data  
2. Forward Pass:                    
3.    : **Loss** =          
4. Backward Pass: gradient ( ) Loss          
5. **Adam Optimizer**             Loss    
6.       batch  training
7.     Validation set

#### ModelCheckpoint  

  **callback** (   )              .
```python
monitor='val_accuracy'  #    : val_accuracy
save_best_only=True     #            
```
        epoch         Overfitting  .
    checkpoint      .

#### Learning Rate (LR)  ?

         :
- LR                    
- LR                  
- LR = 0.001 (1e-3)    


In [ ]:
#       CNN  

#        
best_cnn_path = os.path.join(DATA_DIR, 'best_simple_model.keras')

# ModelCheckpoint:              
checkpoint_cnn = ModelCheckpoint(
    filepath=best_cnn_path,
    monitor='val_accuracy',    #    :   validation
    save_best_only=True,       #        
    mode='max',                #   =   (  accuracy)
    verbose=0                  #      
)

print('    ...')
print(f'  epoch: {EPOCHS_CNN}')
print(f'   : {LR_CNN}')
print(f'Batch size: {BATCH_SIZE}')
print('=' * 50)

# model.fit:      
history_cnn = cnn.fit(
    train_ds,                     #    
    validation_data=val_ds,        #     (   )
    epochs=EPOCHS_CNN,             #    
    callbacks=[checkpoint_cnn],    #  
    verbose=1                      #    
)

print('\n     !')
print(f'  val_accuracy: {max(history_cnn.history["val_accuracy"])*100:.2f}%')


###  .           

####          

**   :**
```
Loss                     Accuracy
    Training              Training  
    Validation            Validation  
  [       ]       [       ]
```

**  Overfitting:**
```
Loss                     Accuracy
    Training              Training   (   )
    Validation            Validation   (     )
  [   !]           [   !]
```

  **dataset  ** (       )   Overfitting  .
            dataset      .


In [ ]:
#          

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS_CNN + 1)

#   Loss
ax = axes[0]
ax.plot(epochs_range, history_cnn.history['loss'],
        color='royalblue', linewidth=2, label='Training Loss')
ax.plot(epochs_range, history_cnn.history['val_loss'],
        color='darkorange', linewidth=2, linestyle='--', label='Validation Loss')
ax.set_title('Simple CNN     Loss', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch ( )')
ax.set_ylabel('Loss ( )')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

#   Accuracy
ax = axes[1]
ax.plot(epochs_range, [v*100 for v in history_cnn.history['accuracy']],
        color='forestgreen', linewidth=2, label='Training Acc')
ax.plot(epochs_range, [v*100 for v in history_cnn.history['val_accuracy']],
        color='crimson', linewidth=2, linestyle='--', label='Validation Acc')
ax.set_title('Simple CNN     Accuracy', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch ( )')
ax.set_ylabel('Accuracy (%)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
curve_path = os.path.join(DATA_DIR, 'step2_training_curves.png')
plt.savefig(curve_path, dpi=150)
plt.close()
display(Image(curve_path))

#    
final_train_acc = history_cnn.history['accuracy'][-1] * 100
final_val_acc   = history_cnn.history['val_accuracy'][-1] * 100
gap = final_train_acc - final_val_acc
print(f'Train Accuracy:      {final_train_acc:.2f}%')
print(f'Validation Accuracy: {final_val_acc:.2f}%')
print(f'  (Gap):        {gap:.2f}%')
if gap > 10:
    print('    Overfitting       (  > 10%)')
    print('      dataset            ')


###  .          Test Set

####   Test Set  

Test Set   ** **              .
    Test Set = **   **                    .

####            
    epoch        .
ModelCheckpoint   epoch                .


In [ ]:
#          
best_cnn = keras.models.load_model(best_cnn_path)
print('    CNN    ')

#     test set
test_loss_cnn, test_acc_cnn = best_cnn.evaluate(test_ds, verbose=0)

print('\n' + '=' * 45)
print('     Simple CNN   Test Set   ')
print('=' * 45)
print(f'  Test Loss     : {test_loss_cnn:.4f}')
print(f'  Test Accuracy : {test_acc_cnn*100:.2f}%')
print('=' * 45)
print()
print('  Train Accuracy (  epoch):', f'{history_cnn.history["accuracy"][-1]*100:.2f}%')
print('  Test Accuracy               :', f'{test_acc_cnn*100:.2f}%')
overfit_gap = history_cnn.history['accuracy'][-1]*100 - test_acc_cnn*100
print(f'    Overfitting           : {overfit_gap:.2f}%')


###      

**       :**
-                    
- BatchNormalization  ReLU  Softmax          
- Overfitting                
- K-Fold           Train/Test    

** :**
  CNN   Overfitting     dataset    .
          **U-Net**            .

**   : U-Net            **


---
#  
#    :   U-Net (Encoder-Decoder   Skip Connections)
#  

##     CNN  

        CNN              .
             :

**           .**

                           .

##  : U-Net

U-Net                .
             .

###    : Encoder-Decoder   Skip Connection

```
[ :    ]
     |
  [Encoder:  ]
     |
       MaxPool         (     )
       MaxPool        
       MaxPool   ...
     |
  [Bottleneck:          ]
     |
  [Decoder:  ]
       UpSample          
       UpSample   ...
     |
[ :            ]
```

### Skip Connection:     U-Net

```
Encoder       Bottleneck    Decoder
                      
Block1   Concat   Block1'
   
Block2   Concat   Block2'
   
Block3   Concat   Block3'
   
Block4   Concat   Block4'
   
Bottleneck  
```

**Skip Connection =    **     encoder   decoder.
-       (   )   Block1     Decoder  
- Decoder              

**     **
- CNN  :     (   )          
- U-Net:           Decoder    


###  .          U-Net

#### MaxPooling  

                     :
```
    MaxPool:          MaxPool 2 2:
           
 1  3  2  4          3  4     (        2 2)
 5  7  6  8            7  9  
 
 2  4  9  1  
 6  8  3  2  
 
```
                  (global context)

#### UpSampling ( )  

  MaxPool            :
```
           
 3  4          3  3  4  4  
 7  9            7  7  9  9  
           
```
   :               (Nearest Neighbor Upsampling)

#### Concatenate ( )  

  Feature Map   UpSampling   Skip Connection    :
```
  UpSampling: (H, W, 256)
  Skip Conn:  (H, W, 256)
    Concat: (H, W, 512)             
```
Decoder         (  bottleneck)         (  encoder)  .


In [ ]:
#       U-Net  

from tensorflow.keras import layers

def conv_block(x, num_filters, block_name):
    """
           : Conv   BN   ReLU   Conv   BN   ReLU
    
                  Encoder     Decoder    .
    
     :
        x           :    
        num_filters :      
        block_name  :            
     :       H,W   num_filters  
    """
    x = layers.Conv2D(
        num_filters, 3,             # 3 3 kernel
        padding='same',             #      
        activation='relu',          #    
        name=f'{block_name}_c1'    #      
    )(x)
    x = layers.BatchNormalization(name=f'{block_name}_bn1')(x)
    x = layers.Conv2D(
        num_filters, 3,
        padding='same',
        activation='relu',
        name=f'{block_name}_c2'
    )(x)
    x = layers.BatchNormalization(name=f'{block_name}_bn2')(x)
    return x


def build_unet(input_channels=5, num_classes=6, base_filters=32):
    """
        U-Net      .
    
     :
        input_channels:       (5 = RGB+IR+Elevation)
        num_classes   :       (6)
        base_filters  :       (32   64   128   256   512)
    """
    f = base_filters  #      
    
    inp = keras.Input(shape=(None, None, input_channels), name='input')
    
    #   ENCODER  
    #    : conv_block     skip   MaxPool (   )
    
    e1 = conv_block(inp, f,   'enc1')   # (H,  W,  32)
    p1 = layers.MaxPooling2D(2, name='pool1')(e1)  # (H/2, W/2, 32)
    
    e2 = conv_block(p1,  f*2, 'enc2')   # (H/2, W/2, 64)
    p2 = layers.MaxPooling2D(2, name='pool2')(e2)  # (H/4, W/4, 64)
    
    e3 = conv_block(p2,  f*4, 'enc3')   # (H/4,  W/4,  128)
    p3 = layers.MaxPooling2D(2, name='pool3')(e3)  # (H/8, W/8, 128)
    
    e4 = conv_block(p3,  f*8, 'enc4')   # (H/8,  W/8,  256)
    p4 = layers.MaxPooling2D(2, name='pool4')(e4)  # (H/16, W/16, 256)
    
    #   BOTTLENECK  
    #          
    b = conv_block(p4, f*16, 'bottleneck')   # (H/16, W/16, 512)
    
    #   DECODER  
    #    : UpSampling   Concatenate   skip   conv_block
    
    #     Decoder:
    u4 = layers.UpSampling2D(2, name='up4')(b)           # (H/8,  W/8,  512)
    u4 = layers.Concatenate(name='cat4')([u4, e4])       #   Skip   enc4
    d4 = conv_block(u4, f*8, 'dec4')                     # (H/8,  W/8,  256)
    
    #     Decoder:
    u3 = layers.UpSampling2D(2, name='up3')(d4)          # (H/4,  W/4,  256)
    u3 = layers.Concatenate(name='cat3')([u3, e3])       #   Skip   enc3
    d3 = conv_block(u3, f*4, 'dec3')                     # (H/4,  W/4,  128)
    
    #     Decoder:
    u2 = layers.UpSampling2D(2, name='up2')(d3)          # (H/2,  W/2,  128)
    u2 = layers.Concatenate(name='cat2')([u2, e2])       #   Skip   enc2
    d2 = conv_block(u2, f*2, 'dec2')                     # (H/2,  W/2,  64)
    
    #     Decoder:
    u1 = layers.UpSampling2D(2, name='up1')(d2)          # (H,    W,    64)
    u1 = layers.Concatenate(name='cat1')([u1, e1])       #   Skip   enc1
    d1 = conv_block(u1, f, 'dec1')                       # (H,    W,    32)
    
    #   OUTPUT  
    # Conv 1 1: 32   6   (         )
    out = layers.Conv2D(num_classes, 1,
                        padding='same', activation='softmax',
                        name='output')(d1)  # (H, W, 6)
    
    model = keras.Model(inputs=inp, outputs=out, name='UNet')
    return model


#   U-Net
unet = build_unet(input_channels=5, num_classes=NUM_CLASSES, base_filters=32)
unet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),  # LR     CNN  
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('    U-Net:')
unet.summary()
print(f'\n   : {unet.count_params():,}')
print(f'      CNN   (215K): U-Net {unet.count_params()/215_000:.1f}x    ')


###  .      Learning Rate     U-Net 

U-Net     (~ .          ~   ).

     :
-     (LR  )      
- `1e-4` = 0.0001 (        CNN     `1e-3`  )

###  .      U-Net

        **   **     (RGB + IR + Elevation).
       
-  :    
-  :        
-  :      


In [ ]:
#     Dataset  U-Net   5    

EPOCHS_UNET = 20

print('  Dataset    5   (RGB+IR+Elevation) ...')
train_ds_u = make_dataset(train_files, augment_data=True,  batch_size=BATCH_SIZE, use_elevation=True)
val_ds_u   = make_dataset(val_files,   augment_data=False, batch_size=BATCH_SIZE, use_elevation=True)
test_ds_u  = make_dataset(test_files,  augment_data=False, batch_size=BATCH_SIZE, use_elevation=True)

# Checkpoint     U-Net
best_unet_path = os.path.join(DATA_DIR, 'best_unet_model.keras')
checkpoint_u = ModelCheckpoint(
    filepath=best_unet_path,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=0
)

print(f'    U-Net   {EPOCHS_UNET} epoch ...')
print('(             )')
print('=' * 50)

history_unet = unet.fit(
    train_ds_u,
    validation_data=val_ds_u,
    epochs=EPOCHS_UNET,
    callbacks=[checkpoint_u],
    verbose=1
)

print('\n  U-Net    !')
print(f'  val_accuracy: {max(history_unet.history["val_accuracy"])*100:.2f}%')


In [ ]:
#         CNN       U-Net  

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('  Simple CNN     U-Net', fontsize=14, fontweight='bold')

ep_cnn  = range(1, EPOCHS_CNN+1)
ep_unet = range(1, EPOCHS_UNET+1)

#    : Simple CNN
axes[0,0].plot(ep_cnn, history_cnn.history['loss'],     label='Train', color='royalblue', lw=2)
axes[0,0].plot(ep_cnn, history_cnn.history['val_loss'], label='Val',   color='darkorange', lw=2, ls='--')
axes[0,0].set_title('Simple CNN   Loss', fontweight='bold')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(ep_cnn, [v*100 for v in history_cnn.history['accuracy']],     label='Train', color='forestgreen', lw=2)
axes[0,1].plot(ep_cnn, [v*100 for v in history_cnn.history['val_accuracy']], label='Val',   color='crimson', lw=2, ls='--')
axes[0,1].set_title('Simple CNN   Accuracy', fontweight='bold')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy (%)')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

#    : U-Net
axes[1,0].plot(ep_unet, history_unet.history['loss'],     label='Train', color='royalblue', lw=2)
axes[1,0].plot(ep_unet, history_unet.history['val_loss'], label='Val',   color='darkorange', lw=2, ls='--')
axes[1,0].set_title('U-Net   Loss', fontweight='bold')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Loss')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(ep_unet, [v*100 for v in history_unet.history['accuracy']],     label='Train', color='forestgreen', lw=2)
axes[1,1].plot(ep_unet, [v*100 for v in history_unet.history['val_accuracy']], label='Val',   color='crimson', lw=2, ls='--')
axes[1,1].set_title('U-Net   Accuracy', fontweight='bold')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Accuracy (%)')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
compare_path = os.path.join(DATA_DIR, 'step3_training_curves.png')
plt.savefig(compare_path, dpi=150)
plt.close()
display(Image(compare_path))


###  .        U-Net

             .

**Argmax  **
  U-Net      : `[0.05, 0.01, 0.02, 0.10, 0.80, 0.02]` (  6  )
`argmax`         = `4` ( )

```python
pred_label = np.argmax(pred_logit, axis=-1)
# (H, W, 6)   (H, W)      :    
```


In [ ]:
#       U-Net      

best_unet = keras.models.load_model(best_unet_path)
test_loss_u, test_acc_u = best_unet.evaluate(test_ds_u, verbose=0)

print('  U-Net   Test Set:')
print(f'  Test Loss    : {test_loss_u:.4f}')
print(f'  Test Accuracy: {test_acc_u*100:.2f}%')

#            

test_fp = test_files[0]
X5_test, _ = load_sample(test_fp, use_elevation=True)

#  :     batch   (1,H,W,5)
X5_batch = np.expand_dims(X5_test, axis=0)  #  : (1, H, W, 5)
pred_probs = best_unet.predict(X5_batch, verbose=0)[0]  #  : (H, W, 6)

# argmax:     =    
pred_label = np.argmax(pred_probs, axis=-1)  #  : (H, W)

#        
with rasterio.open(test_fp) as src:
    raw = src.read()

rgb_v    = np.stack([normalize_band(raw[0].astype(np.float32)),
                     normalize_band(raw[1].astype(np.float32)),
                     normalize_band(raw[2].astype(np.float32))], axis=-1)
elev_v   = normalize_band(raw[4].astype(np.float32))
gt_label = raw[5].astype(np.int32)
gt_rgb   = label_to_rgb(gt_label,   CLASS_COLORS)
pred_rgb = label_to_rgb(pred_label, CLASS_COLORS)

patches = [mpatches.Patch(color=[c/255 for c in CLASS_COLORS[i]], label=CLASS_NAMES[i])
           for i in range(NUM_CLASSES)]

#   4    
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle('U-Net:         (Ground Truth)', fontsize=13, fontweight='bold')

axes[0].imshow(rgb_v);    axes[0].set_title('  RGB  ');   axes[0].axis('off')
im = axes[1].imshow(elev_v, cmap='terrain')
axes[1].set_title('   '); axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
axes[2].imshow(gt_rgb);   axes[2].set_title('    (GT)'); axes[2].axis('off')
axes[2].legend(handles=patches, loc='lower right', fontsize=6.5, framealpha=0.9)
axes[3].imshow(pred_rgb); axes[3].set_title('  U-Net');   axes[3].axis('off')
axes[3].legend(handles=patches, loc='lower right', fontsize=6.5, framealpha=0.9)

plt.tight_layout()
pred_path = os.path.join(DATA_DIR, 'step3_prediction_visualization.png')
plt.savefig(pred_path, dpi=150, bbox_inches='tight')
plt.close()
display(Image(pred_path))
print('       ')


---
#          

##        

|   | Simple CNN | U-Net |
|---|---|---|
| **   ** |   (RGB+IR) |   (RGB+IR+Elevation) |
| ** ** | ~ ,  | ~ , ,  |
| **   ** |   |   |
| ** ** | Flat | Encoder-Decoder |
| **Skip Connection** |   |   |
| **   ** |   |   |

##   U-Net   dataset        

              Overfit            .
  dataset     (     ):
- Skip connections            
- U-Net                
-  : **U-Net        **   Test Set  


In [ ]:
#          

print('\n' + '=' * 60)
print('          ')
print('=' * 60)
print(f'{" ":<15}{" ":<12}{"Test Loss":>12}{"Test Acc":>12}')
print('-' * 53)
print(f'{"Simple CNN":<15}{"4 (RGB+IR)":<12}{test_loss_cnn:>12.4f}{test_acc_cnn*100:>11.2f}%')
print(f'{"U-Net":<15}{"5 (+Elev)":<12}{test_loss_u:>12.4f}{test_acc_u*100:>11.2f}%')
print('=' * 60)

winner = 'U-Net' if test_loss_u < test_loss_cnn else 'Simple CNN'
improvement = abs(test_loss_cnn - test_loss_u)
print(f'\n    (Test Loss  ): {winner}')
print(f'    Loss: {improvement:.4f}')


---
##            

###    :
|   |     |
|---|---|
| **Semantic Segmentation** |       |
| **GeoTIFF** |           |
| **Convolution** |                       |
| **BatchNormalization** |           |
| **ReLU** |  : `max(0,x)` |
| **Softmax** |         |
| **K-Fold CV** |             K   |
| **Overfitting** |             |
| **Data Augmentation** |           Overfitting |
| **U-Net** | Encoder-Decoder   Skip Connection       |
| **Skip Connection** |       Encoder   Decoder         |

###       Dataset  :

1. `DATA_DIR`       dataset   Potsdam    
2. notebook          
3.           `> 80%`    
4. U-Net     Simple CNN    

---
*                           .*
*   !*  
